# 71. 같은 학습에서 30에폭과 50에폭을 기다린 결과

Community RTX 3090에서 원본 YOLO26n을 최대 1000에폭까지 학습합니다. 검증 fitness 최고점을 갱신하지 못한 첫 30에폭 시점에 당시 최고 가중치를 복사합니다. 학습은 patience 50까지 이어집니다. 두 가중치와 기존 200/30 최고 가중치를 시간 연결 조건이 같은 Mac CPU에서 비교합니다.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from wlcv.config import ROOT

folder = ROOT / "outputs/plate_patience_3090/20261126"
run = folder / "runs/schedule1000"
review = folder / "review"
record = json.loads((run / "patience.json").read_text())
curve = pd.read_csv(run / "results.csv")
ax = curve.plot(x="epoch", y="metrics/mAP50-95(B)", legend=False, figsize=(9, 3),
                color="#275D76", title="Validation mAP50-95 throughout training")
if record["first_30_epoch"] is not None:
    ax.axvline(record["first_30_epoch"], color="#AE5A26", linestyle="--", label="First 30-epoch wait")
ax.axvline(record["best_weight_epoch"], color="#267153", linestyle=":", label="Final best checkpoint")
ax.set(xlabel="Epoch", ylabel="mAP50-95")
ax.legend()
plt.tight_layout()
plt.show()
display(pd.Series(record))

## 미탐과 실제 좌표 누락

미탐은 동작 중 수동 박스가 있는 270장에서 IoU가 0.50에 미치지 못하거나 출력이 없는 경우입니다. 좌표 누락은 수동 박스가 없는 프레임도 포함해, 동작 중 시간 연결 후에도 중심 좌표를 내지 못한 경우입니다. 최장 길이는 같은 선수의 연속 프레임 안에서만 셉니다.

In [ ]:
summary = pd.read_csv(review / "summary.csv")
athletes = pd.read_csv(review / "athletes.csv")
display(summary.round(3))
display(athletes[["schedule", "video_id", "misses", "gap_frames", "longest_gap", "gap_indices"]])

## 무릎 통과에서 바와 몸의 접촉까지

수동 상승량은 가까운 원판의 수동 허브가 두 시점 사이에 올라간 세로 픽셀 수입니다. 모델은 같은 두 프레임의 박스 중심으로 계산합니다. 표의 오차는 모델 상승량에서 수동 상승량을 뺀 값입니다. 양수는 과대 측정, 음수는 과소 측정입니다. 한쪽 좌표가 없으면 결측으로 표시합니다.

In [ ]:
labels = ["200/30", "1000/30", "1000/50"]
reference = athletes.drop_duplicates("video_id").set_index("video_id")[["knee", "contact", "manual_rise_px"]]
errors = athletes.pivot(index="video_id", columns="schedule", values="rise_error_px")[labels]
display(reference.join(errors).round(2))
ax = errors.plot.bar(figsize=(8, 3.5), rot=15, color=["#596776", "#14748A", "#B86733"])
ax.axhline(0, color="black", linewidth=.8)
ax.set(xlabel="Validation video", ylabel="Model minus manual rise (px)",
       title="Rise from knee pass to contact")
plt.tight_layout()
plt.savefig(review / "rise_by_athlete.png", dpi=150)
plt.show()